In [1]:
# Installing Dependencies
!pip install datasets pandas matplotlib seaborn scikit-learn

In [2]:
# ASRS Dataset
from datasets import load_dataset
ds = load_dataset('elihoole/asrs-aviation-reports')
work = ds['train']


README.md: 0.00B [00:00, ?B/s]

asrs-aviation-reports-train.jsonl:   0%|          | 0.00/270M [00:00<?, ?B/s]

asrs-aviation-reports-validation.jsonl:   0%|          | 0.00/29.9M [00:00<?, ?B/s]

asrs-aviation-reports-test.jsonl:   0%|          | 0.00/33.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/38655 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4295 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4773 [00:00<?, ? examples/s]

In [3]:
# Sanity check
print(f"Loaded {len(work)} reports")
print(f"Sample Anomaly:", work[100]['Events_Anomaly'])

Loaded 38655 reports
Sample Anomaly: Aircraft Equipment Problem Less Severe; Deviation / Discrepancy - Procedural Published Material / Policy; Flight Deck / Cabin / Aircraft Event Smoke / Fire / Fumes / Odor; Ground Event / Encounter Other / Unknown


# Notebook 01: Evaluation Metrics Implementation

**Author:** Ryan Powers  
**Purpose:** Build industry-standard IR metrics for retrieval system comparison  

**Output:** `retrieval_metrics.py` module with tested functions

---

## Why This Matters

Before we can claim "hybrid is better than BM25", we need objective metrics to measure quality.

This notebook implements 5 core metrics:
1. **Recall@K** - Did we find the relevant reports? (safety coverage)
2. **Precision@K** - How much noise in our results? (user experience)
3. **Average Precision** - How well ranked are results? (per query)
4. **MAP** - Overall system quality (industry standard)
5. **nDCG@K** - Ranking quality with position discount (deep retrieval)


In [4]:
import numpy as np
from typing import List, Set, Dict, Tuple
from collections import defaultdict
import json

In [5]:
def recall_at_k(
    retrieved: List[str],  # Ranked list of document IDs from your system
    relevant: Set[str],    # Ground truth: which docs are actually relevant
    k: int                 # Cutoff depth (10, 100, 1000, etc.)
) -> float:
    """
    Calculate Recall@K: fraction of relevant documents found in top K results.

    Args:
        retrieved: Ordered list of document IDs from your retrieval system.
                   Example: ['1234567', '7654321', '9876543', ...]
                   In ASRS context, these are ACN numbers (acn_num_ACN field).

        relevant: Set of ground-truth relevant document IDs.
                  Example: {'1234567', '9876543', '1111111', '2222222'}
                  These are the reports we KNOW are relevant to the query.

        k: Cutoff rank. Only consider top K results.
           Example: k=100 means "look at top 100 results only"

    Returns:
        Float between 0.0 and 1.0
        - 1.0 = Perfect! Found ALL relevant docs in top K
        - 0.5 = Found half of relevant docs
        - 0.0 = Failure! No relevant docs in top K

    Algorithm:
        1. Take top K results from retrieved list
        2. Check how many are in the relevant set
        3. Divide by total number of relevant docs

    Example:
        retrieved = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
        relevant = {'B', 'D', 'F', 'Z'}  # 4 relevant docs total

        recall_at_k(retrieved, relevant, k=4):
          - Top 4: ['A', 'B', 'C', 'D']
          - Found: B and D (2 out of 4)
          - Recall@4 = 2/4 = 0.5

        recall_at_k(retrieved, relevant, k=8):
          - Top 8: all of them
          - Found: B, D, F (3 out of 4, Z not in retrieved list)
          - Recall@8 = 3/4 = 0.75

    Notes:
        - Maximum achievable recall depends on retrieved list contents
        - If some relevant docs aren't in retrieved at all, max recall < 1.0
        - This is normal in retrieval systems
    """
    # Edge case: If no relevant docs exist, recall is undefined
    # Return 0.0 by convention (can't recall anything if nothing is relevant)
    if len(relevant) == 0:
        return 0.0

    # STEP 1: Get top K retrieved docs
    # List slicing retrieved[:k] gives us first k elements
    # Convert to set for O(1) intersection operation
    retrieved_at_k = set(retrieved[:k])

    # STEP 2: Find intersection (which retrieved docs are relevant?)
    # Set intersection gives us docs that appear in BOTH sets
    found = retrieved_at_k.intersection(relevant)

    # STEP 3: Calculate recall
    # Divide number found by total number relevant
    return len(found) / len(relevant)


print("recall_at_k() function defined")


recall_at_k() function defined


In [6]:

# SCENARIO: Analyst searches "altitude deviation near airport"
# System returns 20 results (these would be ACN numbers in real system)
test_retrieved = [
    'report_102',  # Rank 1
    'report_215',  # Rank 2  - RELEVANT ✓
    'report_403',  # Rank 3
    'report_519',  # Rank 4
    'report_627',  # Rank 5  - RELEVANT ✓
    'report_734',  # Rank 6
    'report_841',  # Rank 7  - RELEVANT ✓
    'report_952',  # Rank 8
    'report_168',  # Rank 9
    'report_273',  # Rank 10
    'report_384',  # Rank 11
    'report_495',  # Rank 12 - RELEVANT ✓
    'report_501',  # Rank 13
    'report_617',  # Rank 14
    'report_728',  # Rank 15 - RELEVANT ✓
    'report_839',  # Rank 16
    'report_944',  # Rank 17
    'report_155',  # Rank 18
    'report_266',  # Rank 19
    'report_377'   # Rank 20
]

# GROUND TRUTH: We determined these 7 reports are relevant
# (In real project, from anomaly field or manual labeling)
test_relevant = {
    'report_215',   # Found at rank 2 ✓
    'report_627',   # Found at rank 5 ✓
    'report_841',   # Found at rank 7 ✓
    'report_495',   # Found at rank 12 ✓
    'report_728',   # Found at rank 15 ✓
    'report_999',   # NOT in retrieved list ✗
    'report_888'    # NOT in retrieved list ✗
}

print(f"Query: 'altitude deviation near airport'")
print(f"Total relevant reports in database: {len(test_relevant)}")
print(f"System returned: {len(test_retrieved)} results\n")

# Calculate recall at different K values
print("Recall at different depths:\n")

for k in [5, 10, 15, 20]:
    r = recall_at_k(test_retrieved, test_relevant, k)

    # Count how many we actually found
    found = set(test_retrieved[:k]).intersection(test_relevant)
    found_count = len(found)
    total_count = len(test_relevant)

    print(f"  Recall@{k:2d} = {r:.3f}  ({found_count}/{total_count} relevant reports found)")

    if k == 5:
        print(f"            → Only looked at top 5, found 2 relevant reports")
    elif k == 15:
        print(f"            → Looked at top 15, found all 5 that are in retrieved list")

print("\n" + "="*80)

print()
print("1. Recall INCREASES as K increases (monotonic property)")
print("   → The deeper you look, the more relevant docs you find")
print()
print("2. Recall@20 = 0.714 (not 1.0) because:")
print("   → System didn't return report_999 and report_888 at all")
print("   → Missing 2 out of 7 relevant reports = 28% coverage gap")
print()
print("3. For safety analysis, this gap is concerning:")
print("   → Missing 28% of relevant incidents = incomplete pattern analysis")
print("   → This is WHY we need hybrid retrieval to improve recall")
print()
print("✓ Recall@K test PASSED")
print()

Query: 'altitude deviation near airport'
Total relevant reports in database: 7
System returned: 20 results

Recall at different depths:

  Recall@ 5 = 0.286  (2/7 relevant reports found)
            → Only looked at top 5, found 2 relevant reports
  Recall@10 = 0.429  (3/7 relevant reports found)
  Recall@15 = 0.714  (5/7 relevant reports found)
            → Looked at top 15, found all 5 that are in retrieved list
  Recall@20 = 0.714  (5/7 relevant reports found)


1. Recall INCREASES as K increases (monotonic property)
   → The deeper you look, the more relevant docs you find

2. Recall@20 = 0.714 (not 1.0) because:
   → System didn't return report_999 and report_888 at all
   → Missing 2 out of 7 relevant reports = 28% coverage gap

3. For safety analysis, this gap is concerning:
   → Missing 28% of relevant incidents = incomplete pattern analysis
   → This is WHY we need hybrid retrieval to improve recall

✓ Recall@K test PASSED



In [7]:
def precision_at_k(
    retrieved: List[str],
    relevant: Set[str],
    k: int
) -> float:
    """
    Calculate Precision@K: fraction of top K that are relevant.

    Args:
        retrieved: Ranked list of doc IDs
        relevant: Ground truth relevant docs
        k: Cutoff

    Returns:
        Float between 0.0 and 1.0
        - 1.0 = Perfect! All K results are relevant
        - 0.5 = Half of K results are relevant
        - 0.0 = None of K results are relevant

    Algorithm:
        1. Take top K results
        2. Count how many are in relevant set
        3. Divide by K

    Example:
        retrieved = ['A', 'B', 'C', 'D', 'E']
        relevant = {'B', 'D', 'F'}

        precision_at_k(retrieved, relevant, k=5):
          - Top 5: ['A', 'B', 'C', 'D', 'E']
          - Relevant: B and D (2 out of 5)
          - Precision@5 = 2/5 = 0.4

    Interpretation:
        "If I show user the top K results, what fraction are useful?"

    Note:
        Unlike recall, precision denominator is K (not len(relevant)).
        This measures the quality of what you returned, not coverage.
    """
    # Edge case: can't have precision if K=0
    if k == 0:
        return 0.0

    # Get top K and find intersection with relevant set
    retrieved_at_k = set(retrieved[:k])
    found = retrieved_at_k.intersection(relevant)

    # Divide by K (not by len(relevant) like recall!)
    return len(found) / k


print("✓ precision_at_k() function defined")

✓ precision_at_k() function defined


In [8]:


print("TEST: Precision@K (Observing Recall/Precision Tradeoff)")

print()

print(f"Using same data as Recall@K test\n")

# Compare Recall and Precision at different K values
print("K    Recall    Precision   Interpretation")
print("-" * 80)

for k in [5, 10, 15, 20]:
    r = recall_at_k(test_retrieved, test_relevant, k)
    p = precision_at_k(test_retrieved, test_relevant, k)

    # Interpretation
    if k == 5:
        interp = "High precision (40%), but missed 71% of relevant docs"
    elif k == 10:
        interp = "Balanced, but still missing some relevant reports"
    elif k == 15:
        interp = "Good recall (71%), precision starting to drop"
    else:
        interp = "Maximum recall for this result set, lowest precision"

    print(f"{k:2d}   {r:.3f}     {p:.3f}       {interp}")


print("1. As K increases:")
print("   → Recall goes UP (finding more relevant docs)")
print("   → Precision goes DOWN (more irrelevant docs mixed in)")
print()
print("2. This is the fundamental retrieval tradeoff:")
print("   → Cast wide net (high K) = better coverage but more noise")
print("   → Narrow focus (low K) = less noise but miss relevant docs")
print()
print("3. For aviation safety, we prioritize Recall:")
print("   → Better to review some irrelevant reports than miss critical incidents")
print("   → But we still want to improve precision (reduce analyst workload)")
print()
print("4. Goal of hybrid retrieval:")
print("   → Higher recall than BM25 (find more relevant incidents)")
print("   → Maintain or improve precision (don't add too much noise)")
print()
print("✓ Precision@K test PASSED")
print()

TEST: Precision@K (Observing Recall/Precision Tradeoff)

Using same data as Recall@K test

K    Recall    Precision   Interpretation
--------------------------------------------------------------------------------
 5   0.286     0.400       High precision (40%), but missed 71% of relevant docs
10   0.429     0.300       Balanced, but still missing some relevant reports
15   0.714     0.333       Good recall (71%), precision starting to drop
20   0.714     0.250       Maximum recall for this result set, lowest precision
1. As K increases:
   → Recall goes UP (finding more relevant docs)
   → Precision goes DOWN (more irrelevant docs mixed in)

2. This is the fundamental retrieval tradeoff:
   → Cast wide net (high K) = better coverage but more noise
   → Narrow focus (low K) = less noise but miss relevant docs

3. For aviation safety, we prioritize Recall:
   → Better to review some irrelevant reports than miss critical incidents
   → But we still want to improve precision (reduce analy

In [9]:
def average_precision(
    retrieved: List[str],
    relevant: Set[str]
) -> float:
    """
    Calculate Average Precision: ranking-aware quality metric.

    Args:
        retrieved: Full ranked list (not truncated at K)
        relevant: Ground truth relevant doc IDs

    Returns:
        AP score between 0.0 and 1.0
        - 1.0 = Perfect! All relevant docs ranked first
        - 0.0 = Terrible! No relevant docs found

    Algorithm:
        1. Walk through retrieved list, rank by rank
        2. At each rank where we find a relevant doc:
           - Calculate precision at that rank
           - Add to running sum
        3. Divide sum by total number of relevant docs

    Example:
        retrieved = ['A', 'B', 'C', 'D', 'E', 'F']
        relevant = {'B', 'D', 'F'}

        Step-by-step:
        Rank 1: 'A' → not relevant, skip
        Rank 2: 'B' → RELEVANT! Precision@2 = 1/2 = 0.5
        Rank 3: 'C' → not relevant, skip
        Rank 4: 'D' → RELEVANT! Precision@4 = 2/4 = 0.5
        Rank 5: 'E' → not relevant, skip
        Rank 6: 'F' → RELEVANT! Precision@6 = 3/6 = 0.5

        AP = (0.5 + 0.5 + 0.5) / 3 = 0.5

    Key Insight:
        If we had ranked relevant docs first: ['B', 'D', 'F', ...]
        Rank 1: 'B' → Precision@1 = 1/1 = 1.0
        Rank 2: 'D' → Precision@2 = 2/2 = 1.0
        Rank 3: 'F' → Precision@3 = 3/3 = 1.0
        AP = (1.0 + 1.0 + 1.0) / 3 = 1.0 (perfect!)

    Notes:
        - AP is sensitive to position: earlier is better
        - Used for per-query analysis
        - Average of AP across all queries = MAP
    """
    # Edge case
    if len(relevant) == 0:
        return 0.0

    score = 0.0
    num_hits = 0  # Running count of relevant docs found so far

    # Walk through ranking
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant:
            # Found a relevant doc at this rank!
            num_hits += 1

            # Calculate precision at this rank
            # Precision@rank = (relevant docs found so far) / (current rank)
            precision_at_rank = num_hits / rank

            # Add to score
            score += precision_at_rank

    # Average over all relevant docs
    # (Even if we didn't find some, they still count in denominator)
    return score / len(relevant)


print("✓ average_precision() function defined")

✓ average_precision() function defined


In [10]:

print("TEST: Average Precision (Position Matters!)")

# Compare two rankings with same recall but different AP
ranking_good = [
    'r1',   # RELEVANT ✓ - Rank 1
    'r2',   # not relevant
    'r3',   # RELEVANT ✓ - Rank 3
    'r4',   # not relevant
    'r5',   # RELEVANT ✓ - Rank 5
    'r6',   # not relevant
    'r7',   # RELEVANT ✓ - Rank 7
    'r8',   # not relevant
]

ranking_bad = [
    'r10',  # not relevant
    'r11',  # not relevant
    'r12',  # not relevant
    'r13',  # not relevant
    'r1',   # RELEVANT ✓ - Rank 5 (buried!)
    'r14',  # not relevant
    'r3',   # RELEVANT ✓ - Rank 7
    'r15',  # not relevant
    'r5',   # RELEVANT ✓ - Rank 9
    'r16',  # not relevant
    'r17',  # not relevant
    'r7',   # RELEVANT ✓ - Rank 12
]

ground_truth = {'r1', 'r3', 'r5', 'r7'}  # 4 relevant docs

print(f"Ground truth: {ground_truth}")
print(f"Total relevant: {len(ground_truth)}\n")

# Walkthrough for good ranking
print("GOOD RANKING (relevant docs appear early):\n")
num_hits = 0
ap_components_good = []

for rank, doc in enumerate(ranking_good, start=1):
    if doc in ground_truth:
        num_hits += 1
        prec = num_hits / rank
        ap_components_good.append(prec)
        print(f"  Rank {rank}: {doc} → RELEVANT ✓ | Precision@{rank} = {num_hits}/{rank} = {prec:.3f}")
    else:
        print(f"  Rank {rank}: {doc} → not relevant")

ap_good = sum(ap_components_good) / len(ground_truth)
print(f"\nAP = ({' + '.join([f'{x:.3f}' for x in ap_components_good])}) / {len(ground_truth)} = {ap_good:.3f}")

# Walkthrough for bad ranking
print("\n" + "-"*80)
print("\nBAD RANKING (same relevant docs, but buried deeper):\n")
num_hits = 0
ap_components_bad = []

for rank, doc in enumerate(ranking_bad, start=1):
    if doc in ground_truth:
        num_hits += 1
        prec = num_hits / rank
        ap_components_bad.append(prec)
        print(f"  Rank {rank}: {doc} → RELEVANT ✓ | Precision@{rank} = {num_hits}/{rank} = {prec:.3f}")
    elif rank <= 12:  # Only show first 12 for brevity
        print(f"  Rank {rank}: {doc} → not relevant")

ap_bad = sum(ap_components_bad) / len(ground_truth)
print(f"\nAP = ({' + '.join([f'{x:.3f}' for x in ap_components_bad])}) / {len(ground_truth)} = {ap_bad:.3f}")

# Compare

recall_good = recall_at_k(ranking_good, ground_truth, k=20)
recall_bad = recall_at_k(ranking_bad, ground_truth, k=20)

print(f"\nGood Ranking: AP = {ap_good:.3f}, Recall@20 = {recall_good:.3f}")
print(f"Bad Ranking:  AP = {ap_bad:.3f}, Recall@20 = {recall_bad:.3f}")
print(f"\nBoth found all 4 relevant docs (same recall)")
print(f"But Good Ranking has {((ap_good/ap_bad - 1) * 100):.1f}% higher AP!")
print()
print("Why AP matters:")
print("  → Recall only cares IF you found relevant docs")
print("  → AP cares HOW WELL you ranked them")
print("  → Better ranking = faster insights for analysts")
print()
print("✓ Average Precision test PASSED")
print()

TEST: Average Precision (Position Matters!)
Ground truth: {'r5', 'r3', 'r7', 'r1'}
Total relevant: 4

GOOD RANKING (relevant docs appear early):

  Rank 1: r1 → RELEVANT ✓ | Precision@1 = 1/1 = 1.000
  Rank 2: r2 → not relevant
  Rank 3: r3 → RELEVANT ✓ | Precision@3 = 2/3 = 0.667
  Rank 4: r4 → not relevant
  Rank 5: r5 → RELEVANT ✓ | Precision@5 = 3/5 = 0.600
  Rank 6: r6 → not relevant
  Rank 7: r7 → RELEVANT ✓ | Precision@7 = 4/7 = 0.571
  Rank 8: r8 → not relevant

AP = (1.000 + 0.667 + 0.600 + 0.571) / 4 = 0.710

--------------------------------------------------------------------------------

BAD RANKING (same relevant docs, but buried deeper):

  Rank 1: r10 → not relevant
  Rank 2: r11 → not relevant
  Rank 3: r12 → not relevant
  Rank 4: r13 → not relevant
  Rank 5: r1 → RELEVANT ✓ | Precision@5 = 1/5 = 0.200
  Rank 6: r14 → not relevant
  Rank 7: r3 → RELEVANT ✓ | Precision@7 = 2/7 = 0.286
  Rank 8: r15 → not relevant
  Rank 9: r5 → RELEVANT ✓ | Precision@9 = 3/9 = 0.333
  R

In [11]:
def mean_average_precision(
    results: Dict[str, List[str]],      # {query_id: [ranked_doc_ids]}
    relevance: Dict[str, Set[str]]      # {query_id: {relevant_doc_ids}}
) -> float:
    """
    Calculate Mean Average Precision across multiple queries.

    THIS IS THE PRIMARY METRIC FOR YOUR PRESENTATION.

    Args:
        results: Dictionary mapping query IDs to retrieval results.
                 Example:
                 {
                   'altitude_deviation': ['1234567', '7654321', ...],
                   'runway_incursion': ['9876543', '1111111', ...],
                   ...
                 }

        relevance: Dictionary mapping query IDs to ground truth.
                   Example:
                   {
                     'altitude_deviation': {'1234567', '9999999', ...},
                     'runway_incursion': {'9876543', '2222222', ...},
                     ...
                   }

    Returns:
        MAP score between 0.0 and 1.0

    Algorithm:
        1. For each query, calculate AP
        2. Average all AP scores

    Usage (Week 2):
        # After running BM25 on 120 test queries:
        bm25_results = {
            'query_001': [...ranked results...],
            'query_002': [...],
            ...
        }

        map_score = mean_average_precision(bm25_results, ground_truth)
        print(f"BM25 MAP: {map_score:.3f}")

        # Then run Dense and compare:
        dense_results = {...}
        dense_map = mean_average_precision(dense_results, ground_truth)
        print(f"Dense MAP: {dense_map:.3f}")

        # Statistical significance test:
        if dense_map > map_score:
            improvement = (dense_map / map_score - 1) * 100
            print(f"Dense improves MAP by {improvement:.1f}%")

    Presentation:
        This single number is your main claim:
        "Hybrid retrieval achieves MAP=0.78, a 20% improvement over
         BM25-only (MAP=0.65) with statistical significance p<0.001."
    """
    if len(results) == 0:
        return 0.0

    ap_scores = []

    # Calculate AP for each query
    for query_id in results:
        if query_id in relevance:
            retrieved = results[query_id]
            relevant = relevance[query_id]
            ap = average_precision(retrieved, relevant)
            ap_scores.append(ap)

    # Return mean
    return np.mean(ap_scores) if ap_scores else 0.0


print("✓ mean_average_precision() function defined")

✓ mean_average_precision() function defined


In [12]:
# Simulate evaluation on 5 different aviation safety queries
test_results = {
    'altitude_violation': [
        'r1', 'r2', 'r3', 'r4', 'r5', 'r6', 'r7', 'r8', 'r9', 'r10'
    ],
    'runway_incursion': [
        'r20', 'r21', 'r22', 'r23', 'r24', 'r25', 'r26', 'r27'
    ],
    'communication_error': [
        'r30', 'r31', 'r32', 'r33', 'r34', 'r35', 'r36', 'r37', 'r38'
    ],
    'fuel_issue': [
        'r40', 'r41', 'r42', 'r43', 'r44', 'r45'
    ],
    'bird_strike': [
        'r50', 'r51', 'r52', 'r53', 'r54', 'r55', 'r56', 'r57'
    ]
}

test_relevance_map = {
    'altitude_violation': {'r2', 'r4', 'r8', 'r11'},         # 4 relevant
    'runway_incursion': {'r20', 'r22', 'r24', 'r26', 'r28'}, # 5 relevant
    'communication_error': {'r31', 'r33', 'r37', 'r39'},     # 4 relevant
    'fuel_issue': {'r40', 'r42', 'r44'},                    # 3 relevant
    'bird_strike': {'r51', 'r53', 'r55', 'r57', 'r59'}      # 5 relevant
}

print("Evaluating system on 5 queries:\n")

individual_aps = []
for qid in test_results:
    retrieved = test_results[qid]
    relevant = test_relevance_map[qid]
    ap = average_precision(retrieved, relevant)
    individual_aps.append(ap)

    # Stats for this query
    found = len(set(retrieved).intersection(relevant))
    total = len(relevant)
    recall = found / total if total > 0 else 0

    print(f"  {qid:25s} | AP={ap:.3f} | Recall={recall:.3f} | Found {found}/{total}")

# Calculate MAP
map_score = mean_average_precision(test_results, test_relevance_map)


print(f"MEAN AVERAGE PRECISION (MAP) = {map_score:.3f}")


print()
print(f"This MAP={map_score:.3f} represents average retrieval quality across all queries.")
print()
print("In your project (Week 2-3), you'll compute MAP for:")
print(f"  1. BM25 baseline:    MAP = 0.65 (hypothetical)")
print(f"  2. Dense baseline:   MAP = 0.70 (hypothetical)")
print(f"  3. Hybrid system:    MAP = 0.78 (hypothetical)")
print()
print("Then you'll present:")
print('  "Hybrid retrieval achieves MAP=0.78, a 20% improvement over')
print('   BM25-only (MAP=0.65) with statistical significance p<0.001."')
print()
print("This single number (MAP) is THE PROOF your project needs!")
print()
print("✓ MAP test PASSED")
print()

Evaluating system on 5 queries:

  altitude_violation        | AP=0.344 | Recall=0.750 | Found 3/4
  runway_incursion          | AP=0.568 | Recall=0.800 | Found 4/5
  communication_error       | AP=0.344 | Recall=0.750 | Found 3/4
  fuel_issue                | AP=0.756 | Recall=1.000 | Found 3/3
  bird_strike               | AP=0.400 | Recall=0.800 | Found 4/5
MEAN AVERAGE PRECISION (MAP) = 0.482

This MAP=0.482 represents average retrieval quality across all queries.

In your project (Week 2-3), you'll compute MAP for:
  1. BM25 baseline:    MAP = 0.65 (hypothetical)
  2. Dense baseline:   MAP = 0.70 (hypothetical)
  3. Hybrid system:    MAP = 0.78 (hypothetical)

Then you'll present:
  "Hybrid retrieval achieves MAP=0.78, a 20% improvement over
   BM25-only (MAP=0.65) with statistical significance p<0.001."

This single number (MAP) is THE PROOF your project needs!

✓ MAP test PASSED



In [13]:
def dcg_at_k(
    retrieved: List[str],
    relevant: Set[str],
    k: int
) -> float:
    """
    Calculate Discounted Cumulative Gain at K.

    Args:
        retrieved: Ranked list
        relevant: Ground truth
        k: Cutoff

    Returns:
        DCG score (higher = better)

    Algorithm:
        For each position i in top K:
            If doc is relevant:
                Add (1.0 / log2(position + 1)) to score

    Example:
        retrieved = ['A', 'B', 'C', 'D']
        relevant = {'B', 'D'}

        DCG@4:
        Rank 1: 'A' not relevant → contributes 0
        Rank 2: 'B' relevant → contributes 1.0/log2(3) = 0.631
        Rank 3: 'C' not relevant → contributes 0
        Rank 4: 'D' relevant → contributes 1.0/log2(5) = 0.431

        DCG@4 = 0.631 + 0.431 = 1.062
    """
    dcg = 0.0

    for i, doc_id in enumerate(retrieved[:k], start=1):
        if doc_id in relevant:
            # Relevant doc found!
            gain = 1.0  # Binary relevance (could be graded in advanced use)
            discount = 1.0 / np.log2(i + 1)  # Logarithmic position discount
            dcg += gain * discount

    return dcg


def ndcg_at_k(
    retrieved: List[str],
    relevant: Set[str],
    k: int
) -> float:
    """
    Calculate Normalized DCG: DCG normalized by ideal DCG.

    Args:
        retrieved: Actual ranking
        relevant: Ground truth
        k: Cutoff

    Returns:
        nDCG between 0.0 and 1.0
        - 1.0 = Perfect ranking (all relevant docs ranked first)
        - 0.0 = No relevant docs in top K

    Normalization:
        Divide actual DCG by "ideal DCG" (IDCG)
        IDCG = DCG if we ranked all relevant docs first

        This makes nDCG comparable across queries with
        different numbers of relevant docs.

    Example:
        Query A: 5 relevant docs → IDCG_A
        Query B: 20 relevant docs → IDCG_B

        Without normalization, Query B would have higher DCG
        just because more relevant docs exist.

        With normalization, both can achieve nDCG=1.0 if
        they rank their relevant docs optimally.

    Usage:
        For deep retrieval in aviation safety, use nDCG@1000
        to measure ranking quality across entire result set.
    """
    # Calculate actual DCG
    actual_dcg = dcg_at_k(retrieved, relevant, k)

    # Calculate ideal DCG (all relevant docs ranked first)
    # Build ideal ranking: relevant docs first, then non-relevant
    ideal_retrieved = list(relevant) + [d for d in retrieved if d not in relevant]
    ideal_dcg = dcg_at_k(ideal_retrieved, relevant, k)

    # Normalize
    if ideal_dcg == 0:
        return 0.0

    return actual_dcg / ideal_dcg


print("✓ dcg_at_k() and ndcg_at_k() functions defined")

✓ dcg_at_k() and ndcg_at_k() functions defined


In [14]:

# Compare two rankings: same recalls, different positions
ranking_early = ['r1', 'r2', 'r3', 'r4', 'r5', 'r6', 'r7', 'r8', 'r9', 'r10']
ranking_late = ['r6', 'r7', 'r8', 'r9', 'r10', 'r1', 'r2', 'r3', 'r4', 'r5']
ground_truth = {'r2', 'r4', 'r6', 'r8'}  # 4 relevant

print(f"Ground truth: {ground_truth}")
print(f"Total relevant: {len(ground_truth)}\n")

# Show discount values
print("Position discount values (shows penalty for low ranks):\n")
for i in [1, 2, 3, 5, 10, 20, 50, 100, 1000]:
    discount = 1.0 / np.log2(i + 1)
    penalty = (1.0 - discount) * 100
    print(f"  Rank {i:4d}: discount={discount:.3f} (penalty={penalty:.0f}%)")

print("\n" + "-"*80)
print("\nRANKING A (relevant docs at early positions):\n")
for i, doc in enumerate(ranking_early, start=1):
    marker = "✓" if doc in ground_truth else " "
    discount = 1.0 / np.log2(i + 1)
    contribution = discount if doc in ground_truth else 0.0
    print(f"  Rank {i:2d}: {doc} {marker}  discount={discount:.3f}  contrib={contribution:.3f}")

ndcg_early = ndcg_at_k(ranking_early, ground_truth, k=10)
recall_early = recall_at_k(ranking_early, ground_truth, k=10)
print(f"\nnDCG@10 = {ndcg_early:.3f}  |  Recall@10 = {recall_early:.3f}")

print("\n" + "-"*80)
print("\nRANKING B (same relevant docs, but at later positions):\n")
for i, doc in enumerate(ranking_late, start=1):
    marker = "✓" if doc in ground_truth else " "
    discount = 1.0 / np.log2(i + 1)
    contribution = discount if doc in ground_truth else 0.0
    print(f"  Rank {i:2d}: {doc} {marker}  discount={discount:.3f}  contrib={contribution:.3f}")

ndcg_late = ndcg_at_k(ranking_late, ground_truth, k=10)
recall_late = recall_at_k(ranking_late, ground_truth, k=10)
print(f"\nnDCG@10 = {ndcg_late:.3f}  |  Recall@10 = {recall_late:.3f}")


print(f"1. Both rankings found ALL 4 relevant docs:")
print(f"   → Recall@10 = {recall_early:.3f} for both")
print()
print(f"2. But nDCG is very different:")
print(f"   → Ranking A (early): nDCG@10 = {ndcg_early:.3f}")
print(f"   → Ranking B (late):  nDCG@10 = {ndcg_late:.3f}")
print(f"   → Ranking A is {((ndcg_early/ndcg_late - 1)*100):.1f}% better!")
print()
print(f"3. Why? Logarithmic position penalty:")
print(f"   → Relevant doc at rank 2 contributes 0.631")
print(f"   → Relevant doc at rank 8 contributes 0.333")
print(f"   → Early positions worth ~2x more")
print()
print(f"4. This is why nDCG matters:")
print(f"   → MAP and Recall care IF you found relevant docs")
print(f"   → nDCG cares HOW WELL you ranked them")
print(f"   → Better metric for ranking algorithm evaluation")
print()
print(f"5. For deep retrieval (K=1000) in aviation safety:")
print(f"   → nDCG@1000 measures ranking quality across entire result set")
print(f"   → Critical when analysts review 100-1000 reports")
print(f"   → Report alongside MAP in your presentation")
print()
print("✓ nDCG@K test PASSED")
print()

Ground truth: {'r2', 'r8', 'r4', 'r6'}
Total relevant: 4

Position discount values (shows penalty for low ranks):

  Rank    1: discount=1.000 (penalty=0%)
  Rank    2: discount=0.631 (penalty=37%)
  Rank    3: discount=0.500 (penalty=50%)
  Rank    5: discount=0.387 (penalty=61%)
  Rank   10: discount=0.289 (penalty=71%)
  Rank   20: discount=0.228 (penalty=77%)
  Rank   50: discount=0.176 (penalty=82%)
  Rank  100: discount=0.150 (penalty=85%)
  Rank 1000: discount=0.100 (penalty=90%)

--------------------------------------------------------------------------------

RANKING A (relevant docs at early positions):

  Rank  1: r1    discount=1.000  contrib=0.000
  Rank  2: r2 ✓  discount=0.631  contrib=0.631
  Rank  3: r3    discount=0.500  contrib=0.000
  Rank  4: r4 ✓  discount=0.431  contrib=0.431
  Rank  5: r5    discount=0.387  contrib=0.000
  Rank  6: r6 ✓  discount=0.356  contrib=0.356
  Rank  7: r7    discount=0.333  contrib=0.000
  Rank  8: r8 ✓  discount=0.315  contrib=0.315
  

In [15]:
def evaluate_retrieval(
    results: Dict[str, List[str]],
    relevance: Dict[str, Set[str]],
    k_values: List[int] = [10, 50, 100, 1000]
) -> Dict:
    """
    Comprehensive evaluation across all metrics.

    THIS IS THE MAIN FUNCTION YOU'LL USE IN WEEKS 2-3.

    Args:
        results: {query_id: [ranked_doc_ids]} from your retrieval system
        relevance: {query_id: {relevant_doc_ids}} ground truth
        k_values: List of K values to evaluate at

    Returns:
        Dictionary with ALL metrics:
        {
            'map': 0.78,  # Mean Average Precision
            'recall': {
                10: 0.45,
                50: 0.72,
                100: 0.81,
                1000: 0.92
            },
            'precision': {
                10: 0.85,
                50: 0.62,
                100: 0.48,
                1000: 0.15
            },
            'ndcg': {
                10: 0.88,
                50: 0.82,
                100: 0.79,
                1000: 0.76
            }
        }

    Usage Example (Notebook 04 - BM25 Baseline Evaluation):
        # After running BM25 on all 120 test queries:
        bm25_metrics = evaluate_retrieval(
            bm25_results,
            ground_truth_relevance
        )

        print(f"BM25 MAP: {bm25_metrics['map']:.3f}")
        print(f"BM25 Recall@1000: {bm25_metrics['recall'][1000]:.3f}")
        print(f"BM25 nDCG@1000: {bm25_metrics['ndcg'][1000]:.3f}")

        # Then run hybrid and compare:
        hybrid_metrics = evaluate_retrieval(
            hybrid_results,
            ground_truth_relevance
        )

        improvement = (hybrid_metrics['map'] / bm25_metrics['map'] - 1) * 100
        print(f"Hybrid improves MAP by {improvement:.1f}%")

    Presentation:
        Create comparison table:

        System       | Recall@1000 | MAP  | nDCG@1000
        -------------|-------------|------|----------
        BM25 only    | 0.72        | 0.65 | 0.68
        Dense only   | 0.68        | 0.70 | 0.71
        Hybrid       | 0.85        | 0.78 | 0.80  ← WINNER
    """
    metrics = {
        'map': mean_average_precision(results, relevance),
        'recall': {},
        'precision': {},
        'ndcg': {}
    }

    # Calculate metrics at each K value
    for k in k_values:
        recall_scores = []
        precision_scores = []
        ndcg_scores = []

        for qid in results:
            if qid in relevance:
                retrieved = results[qid]
                relevant = relevance[qid]

                recall_scores.append(recall_at_k(retrieved, relevant, k))
                precision_scores.append(precision_at_k(retrieved, relevant, k))
                ndcg_scores.append(ndcg_at_k(retrieved, relevant, k))

        # Average across all queries
        metrics['recall'][k] = np.mean(recall_scores) if recall_scores else 0.0
        metrics['precision'][k] = np.mean(precision_scores) if precision_scores else 0.0
        metrics['ndcg'][k] = np.mean(ndcg_scores) if ndcg_scores else 0.0

    return metrics


print("✓ evaluate_retrieval() function defined")

✓ evaluate_retrieval() function defined


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [16]:


# Use the multi-query test data from MAP test
all_metrics = evaluate_retrieval(
    test_results,
    test_relevance_map,
    k_values=[5, 10, 20]
)

print("RESULTS FOR 5 TEST QUERIES:\n")

print(f"Mean Average Precision (MAP): {all_metrics['map']:.3f}")
print(f"  → Primary metric for system comparison")
print()

print("Recall@K (coverage at different depths):")
for k, score in sorted(all_metrics['recall'].items()):
    print(f"  K={k:2d}  →  {score:.3f}")
print()

print("Precision@K (quality at different depths):")
for k, score in sorted(all_metrics['precision'].items()):
    print(f"  K={k:2d}  →  {score:.3f}")
print()

print("nDCG@K (ranking quality at different depths):")
for k, score in sorted(all_metrics['ndcg'].items()):
    print(f"  K={k:2d}  →  {score:.3f}")




RESULTS FOR 5 TEST QUERIES:

Mean Average Precision (MAP): 0.482
  → Primary metric for system comparison

Recall@K (coverage at different depths):
  K= 5  →  0.600
  K=10  →  0.820
  K=20  →  0.820

Precision@K (quality at different depths):
  K= 5  →  0.480
  K=10  →  0.340
  K=20  →  0.170

nDCG@K (ranking quality at different depths):
  K= 5  →  0.543
  K=10  →  0.660
  K=20  →  0.660


In [17]:
%%writefile retrieval_metrics.py
"""
Evaluation metrics for information retrieval systems.

Author: Ryan Powers
Project: Aviation Safety Hybrid Retrieval System
Date: February 9, 2026

Metrics implemented:
  - Recall@K: Coverage of relevant documents
  - Precision@K: Quality of top K results
  - Average Precision (AP): Ranking quality per query
  - Mean Average Precision (MAP): Aggregate performance
  - nDCG@K: Normalized ranking quality with position discount
"""

import numpy as np
from typing import List, Set, Dict


def recall_at_k(retrieved: List[str], relevant: Set[str], k: int) -> float:
    """Fraction of relevant docs found in top K."""
    if len(relevant) == 0:
        return 0.0
    retrieved_at_k = set(retrieved[:k])
    hits = retrieved_at_k.intersection(relevant)
    return len(hits) / len(relevant)


def precision_at_k(retrieved: List[str], relevant: Set[str], k: int) -> float:
    """Fraction of top K that are relevant."""
    if k == 0:
        return 0.0
    retrieved_at_k = set(retrieved[:k])
    hits = retrieved_at_k.intersection(relevant)
    return len(hits) / k


def average_precision(retrieved: List[str], relevant: Set[str]) -> float:
    """Ranking-aware quality metric for single query."""
    if len(relevant) == 0:
        return 0.0

    score = 0.0
    num_hits = 0

    for i, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant:
            num_hits += 1
            precision_at_i = num_hits / i
            score += precision_at_i

    return score / len(relevant)


def mean_average_precision(
    results: Dict[str, List[str]],
    relevance: Dict[str, Set[str]]
) -> float:
    """Average of AP across all queries (industry standard)."""
    if len(results) == 0:
        return 0.0

    ap_scores = []
    for query_id in results:
        if query_id in relevance:
            ap = average_precision(results[query_id], relevance[query_id])
            ap_scores.append(ap)

    return np.mean(ap_scores) if ap_scores else 0.0


def dcg_at_k(retrieved: List[str], relevant: Set[str], k: int) -> float:
    """Discounted Cumulative Gain with log discount."""
    dcg = 0.0
    for i, doc_id in enumerate(retrieved[:k], start=1):
        if doc_id in relevant:
            dcg += 1.0 / np.log2(i + 1)
    return dcg


def ndcg_at_k(retrieved: List[str], relevant: Set[str], k: int) -> float:
    """Normalized DCG: standard for deep retrieval evaluation."""
    actual_dcg = dcg_at_k(retrieved, relevant, k)
    ideal_retrieved = list(relevant) + [d for d in retrieved if d not in relevant]
    ideal_dcg = dcg_at_k(ideal_retrieved, relevant, k)

    if ideal_dcg == 0:
        return 0.0
    return actual_dcg / ideal_dcg


def evaluate_retrieval(
    results: Dict[str, List[str]],
    relevance: Dict[str, Set[str]],
    k_values: List[int] = [10, 50, 100, 1000]
) -> Dict:
    """
    Comprehensive evaluation across all metrics.

    Returns dictionary with map, recall, precision, ndcg scores.
    """
    metrics = {
        'map': mean_average_precision(results, relevance),
        'recall': {},
        'precision': {},
        'ndcg': {}
    }

    for k in k_values:
        recall_scores = []
        precision_scores = []
        ndcg_scores = []

        for qid in results:
            if qid in relevance:
                retrieved = results[qid]
                relevant = relevance[qid]

                recall_scores.append(recall_at_k(retrieved, relevant, k))
                precision_scores.append(precision_at_k(retrieved, relevant, k))
                ndcg_scores.append(ndcg_at_k(retrieved, relevant, k))

        metrics['recall'][k] = np.mean(recall_scores) if recall_scores else 0.0
        metrics['precision'][k] = np.mean(precision_scores) if precision_scores else 0.0
        metrics['ndcg'][k] = np.mean(ndcg_scores) if ndcg_scores else 0.0

    return metrics


if __name__ == "__main__":
    print("✓ retrieval_metrics.py module loaded successfully")

Writing retrieval_metrics.py


In [18]:
# Testing the module was created
import os
if os.path.exists('retrieval_metrics.py'):
    print("✓ retrieval_metrics.py created successfully!")
    print(f"  File size: {os.path.getsize('retrieval_metrics.py')} bytes")

    # Test importing it
    from retrieval_metrics import evaluate_retrieval
    print("✓ Module imports correctly!")
    print("\n NOTEBOOK 01 COMPLETE!")
else:
    print("❌ File not created - run the %%writefile cell above")

✓ retrieval_metrics.py created successfully!
  File size: 3998 bytes
✓ Module imports correctly!

 NOTEBOOK 01 COMPLETE!
